# F1 Undercut Strategy Prediction
---
**Research Question:** Given the state of a battle on the lap before a driver pits, can I predict whether stopping early will let that driver jump the car ahead after the rival completes their own stop?

**Approach:**
- **Label:** undercut success = the pitting driver is ahead of the original car ahead 4 laps after that car ahead completes its next pit stop.
- **Decision point:** the lap immediately before the undercutting driver pits (`dec_lap = pit_lap - 1`), so the features represent what was knowable before the stop.
- **Exclusions:** attempts with Safety Car or VSC interference in the resolution window are dropped, because those outcomes are no longer clean undercut decisions.
- **Temporal split:** train 2022-2023, test 2024. This matches the overcut notebook so the two strategy models are evaluated on the same historical logic.

**Pipeline:** this notebook mirrors the `scripts/build_undercut_dataset.py` workflow, using the cached `data/f1_undercut_dataset.csv` when available. It then trains the same two-model comparison used in the overcut analysis: Logistic Regression as an interpretable baseline and XGBoost as the primary non-linear model.

---


## 1. Setup

This section imports the analysis stack, resolves project paths, enables the FastF1 cache, and fixes the shared random seed. The rest of the notebook can then focus on the undercut dataset, model comparison, and interpretation.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for script execution
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Resolve project root (works whether notebook runs from notebooks/ or root) ──
import os
_nb_dir = os.path.abspath('')
ROOT = os.path.dirname(_nb_dir) if os.path.basename(_nb_dir) == 'notebooks' else _nb_dir
DATA_DIR    = os.path.join(ROOT, 'data')
MODEL_DIR   = os.path.join(ROOT, 'models')
FIGURES_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


RANDOM_STATE = 42
plt.style.use('seaborn-v0_8-darkgrid')

print('XGBoost:', xgb.__version__)

XGBoost: 3.2.0


## 2. Load Pre-Built Dataset

Dataset generation is owned by `scripts/build_undercut_dataset.py`. This notebook does not rebuild raw FastF1 records; it loads the script-generated `data/f1_undercut_dataset.csv` and focuses on analysis, EDA, model evaluation, SHAP interpretation, and calibration.

The label still follows the same undercut definition: success means the early-stopping driver is ahead of the original car ahead 4 laps after that rival completes its next pit stop, with Safety Car and VSC-contaminated windows excluded by the build script.


In [2]:
# ─────────────────────────────────────────────
# Artifact paths and temporal split
# ─────────────────────────────────────────────

TRAIN_YEARS      = [2022, 2023]
TEST_YEARS       = [2024]

DATASET_PATH     = os.path.join(DATA_DIR,  'f1_undercut_dataset.csv')
MODEL_PATH       = os.path.join(MODEL_DIR, 'f1_undercut_model.pkl')


In [3]:
# ─────────────────────────────────────────────
# Load script-built undercut dataset
# ─────────────────────────────────────────────

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(
        f'Missing {DATASET_PATH}. Run scripts/build_undercut_dataset.py first.'
    )

print('Loading script-built undercut dataset...')
df = pd.read_csv(DATASET_PATH)

print(f'\nDataset shape: {df.shape}')
print(f'Years covered: {sorted(df["year"].unique())}')
print(f'\nLabel distribution:')
vc = df['undercut_success'].value_counts()
print(f'  Success (1): {vc.get(1, 0):,}  ({vc.get(1,0)/len(df):.1%})')
print(f'  Failure (0): {vc.get(0, 0):,}  ({vc.get(0,0)/len(df):.1%})')


Loading script-built undercut dataset...

Dataset shape: (966, 20)
Years covered: [np.int64(2022), np.int64(2023), np.int64(2024)]

Label distribution:
  Success (1): 278  (28.8%)
  Failure (0): 688  (71.2%)


## 3. Feature Engineering

The model uses features that describe both the immediate battle and the physical tyre/pit-stop context:

- `gap_ahead`: how much time the undercutting driver needs to recover.
- `tire_age` and `car_ahead_tire_age`: absolute tyre ages for both cars.
- `tire_age_advantage`: car-ahead tyre age minus pitting-driver tyre age; larger values mean the target car is on older tyres, which should make the undercut easier.
- `own_pace`, `threat_pace`, and `pace_delta`: recent clean-air pace comparison between the undercutting driver and the car ahead.
- `deg_delta` and `ca_deg_delta`: recent degradation slopes for the two cars.
- `closing_rate`: whether the gap was already shrinking before the stop.
- `pit_loss` and `pit_loss_fraction`: circuit-specific cost of stopping, both absolute and scaled to lap time.
- `race_progress`: where the attempt occurs in the race, since tyres, fuel load, and strategic urgency change over time.
- compound dummies: tyre-compound context without forcing the model to treat compounds as ordinal.

Rows missing the core battle-state fields are dropped, while remaining gaps are median-imputed to keep the feature layout aligned with the saved model contract.


In [4]:
# ─────────────────────────────────────────────
# Feature engineering & cleaning
# ─────────────────────────────────────────────

# One-hot encode compound
compound_dummies = pd.get_dummies(df['compound'], prefix='compound')
df_feat = pd.concat([df, compound_dummies], axis=1)

BASE_FEATURES = [
    'gap_ahead',
    'tire_age',
    'car_ahead_tire_age',
    'tire_age_advantage',
    'own_pace',
    'threat_pace',
    'pace_delta',
    'deg_delta',
    'ca_deg_delta',
    'closing_rate',
    'pit_loss',
    'pit_loss_fraction',
    'race_progress',
]

FEATURES = BASE_FEATURES + [c for c in compound_dummies.columns]

# Drop rows missing the most important features
REQUIRED = ['gap_ahead', 'tire_age', 'car_ahead_tire_age', 'own_pace', 'threat_pace']
df_model = df_feat.dropna(subset=REQUIRED + ['undercut_success']).copy()

# Fill remaining NaNs with column median
for col in FEATURES:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(df_model[col].median())
    else:
        df_model[col] = 0   # compound not present in data → all zeros

print(f'Modelling dataset: {len(df_model):,} samples, {len(FEATURES)} features')
print(f'Class balance: {df_model["undercut_success"].mean():.1%} successful undercuts')
df_model.head(3)

Modelling dataset: 966 samples, 18 features
Class balance: 28.8% successful undercuts


,year,circuit,driver,car_ahead,pit_lap,gap_ahead,tire_age,car_ahead_tire_age,tire_age_advantage,compound,...,closing_rate,pit_loss,pit_loss_fraction,race_progress,undercut_success,compound_HARD,compound_INTERMEDIATE,compound_MEDIUM,compound_SOFT,compound_WET
0,2022,Bahrain Grand Prix,VER,LEC,14,3.472,16.0,13.0,-3.0,SOFT,...,0.1961,25.145,0.254500,0.228070,0,False,False,False,True,False
1,2022,Bahrain Grand Prix,VER,LEC,30,3.680,15.0,17.0,2.0,SOFT,...,-0.0340,25.145,0.258872,0.508772,0,False,False,False,True,False
2,2022,Bahrain Grand Prix,GAS,MAG,14,1.576,16.0,13.0,-3.0,SOFT,...,-0.0611,25.145,0.250250,0.228070,0,False,False,False,True,False


## 4. Exploratory Data Analysis

### 4.1 Feature distributions split by outcome

The first EDA view overlays successful and failed undercuts for the main numeric features. This is a quick check on whether the variables separate the classes before relying on model output. For undercuts, I expect the clearest separation around relative tyre age, recent pace, gap size, degradation, and race phase.


In [5]:
# ─────────────────────────────────────────────
# EDA — distributions
# ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

eda_features = [
    'gap_ahead', 'tire_age', 'car_ahead_tire_age', 'tire_age_advantage',
    'pace_delta', 'deg_delta', 'closing_rate', 'race_progress'
]
labels_map = {1: 'Success', 0: 'Failure'}
colours    = {1: '#2ecc71', 0: '#e74c3c'}

for ax, feat in zip(axes, eda_features):
    for label, name in labels_map.items():
        subset = df_model[df_model['undercut_success'] == label][feat].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=colours[label], label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Feature Distributions by Undercut Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_eda.png'), dpi=150, bbox_inches='tight')
plt.show()
print('EDA saved to undercut_eda.png')

EDA saved to undercut_eda.png


### 4.2 Gap to car ahead vs undercut success

The undercut has a direct timing constraint: the larger the gap to the car ahead, the more lap time the pitting driver must gain with fresh tyres and pit-cycle timing. Binning `gap_ahead` makes that relationship easier to inspect and also shows how many real attempts exist in each gap range.


In [6]:
# ─────────────────────────────────────────────
# EDA — gap ahead vs success rate (binned)
# ─────────────────────────────────────────────

df_model['gap_bin'] = pd.cut(df_model['gap_ahead'], bins=[0, 2, 4, 6, 8, 12, 20, 30])
gap_success = df_model.groupby('gap_bin', observed=True)['undercut_success'].agg(['mean', 'count'])
gap_success.columns = ['success_rate', 'n_attempts']

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(range(len(gap_success)), gap_success['success_rate'],
        color='steelblue', alpha=0.8, label='Success Rate')
ax2.plot(range(len(gap_success)), gap_success['n_attempts'],
         'o--', color='tomato', label='# Attempts')

ax1.set_xticks(range(len(gap_success)))
ax1.set_xticklabels([str(b) for b in gap_success.index], rotation=30)
ax1.set_xlabel('Gap to Car Ahead (s)')
ax1.set_ylabel('Undercut Success Rate', color='steelblue')
ax2.set_ylabel('Number of Attempts', color='tomato')
ax1.set_title('Undercut Success Rate by Gap to Car Ahead', fontsize=12, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_gap_vs_success.png'), dpi=150, bbox_inches='tight')
plt.show()
print(gap_success.to_string())

          success_rate  n_attempts
gap_bin                           
(0, 2]        0.553846         325
(2, 4]        0.295547         247
(4, 6]        0.119497         159
(6, 8]        0.043478          69
(8, 12]       0.035714          84
(12, 20]      0.000000          61
(20, 30]      0.000000          21


### 4.3 Tire-age advantage vs success

`tire_age_advantage` is defined as the car ahead's tyre age minus the undercutting driver's tyre age. Positive values mean the car ahead is already on older rubber, so the early-stopping driver should have more opportunity to gain time once fresh tyres are fitted. This is the undercut mirror of the overcut notebook's tire-age-delta view.


In [7]:
# ─────────────────────────────────────────────
# EDA — tire age advantage vs success rate
# ─────────────────────────────────────────────

df_model['age_adv_bin'] = pd.cut(
    df_model['tire_age_advantage'],
    bins=[-30, -10, -5, 0, 5, 10, 20, 40]
)
age_success = df_model.groupby('age_adv_bin', observed=True)['undercut_success'].agg(['mean', 'count'])
age_success.columns = ['success_rate', 'n_attempts']

fig, ax = plt.subplots(figsize=(10, 5))
colours_bar = ['#c0392b' if r < 0.5 else '#27ae60' for r in age_success['success_rate']]
ax.bar(range(len(age_success)), age_success['success_rate'],
       color=colours_bar, alpha=0.85)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1, label='50% baseline')
ax.set_xticks(range(len(age_success)))
ax.set_xticklabels([str(b) for b in age_success.index], rotation=30)
ax.set_xlabel('Tire Age Advantage (car_ahead_age − driver_age, laps)')
ax.set_ylabel('Undercut Success Rate')
ax.set_title('Undercut Success Rate by Relative Tire Age Advantage', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_age_vs_success.png'), dpi=150, bbox_inches='tight')
plt.show()
print(age_success.to_string())

             success_rate  n_attempts
age_adv_bin                          
(-30, -10]       0.125000         104
(-10, -5]        0.184211          76
(-5, 0]          0.315024         619
(0, 5]           0.328671         143
(5, 10]          0.615385          13
(10, 20]         0.142857           7
(20, 40]         0.000000           2


## 5. Train / Test Split (Temporal)

The split is temporal rather than random: 2022-2023 is used for training and 2024 is held out for testing. That matters because race strategy changes by season, team, tyre behavior, and field compression. A row-level random split would let near-duplicate race contexts leak across train and test; the 2024 holdout is a cleaner test of whether the model generalizes to later races.


In [8]:
# ─────────────────────────────────────────────
# Train / test split (temporal)
# ─────────────────────────────────────────────

train_mask = df_model['year'].isin(TRAIN_YEARS)
test_mask  = df_model['year'].isin(TEST_YEARS)

X_train = df_model.loc[train_mask, FEATURES].values.astype(float)
y_train = df_model.loc[train_mask, 'undercut_success'].values.astype(int)

X_test  = df_model.loc[test_mask,  FEATURES].values.astype(float)
y_test  = df_model.loc[test_mask,  'undercut_success'].values.astype(int)

print(f'Train (2022–2023): {len(X_train):,} samples  |  '
      f'Success rate: {y_train.mean():.1%}')
print(f'Test  (2024):      {len(X_test):,} samples  |  '
      f'Success rate: {y_test.mean():.1%}')

Train (2022–2023): 630 samples  |  Success rate: 28.3%
Test  (2024):      336 samples  |  Success rate: 29.8%


## 6. Model Evaluation, 2024 Holdout

This section compares two models on the same 2024 holdout. Logistic Regression is the transparent baseline: it is useful for checking whether the signal is mostly linear and monotonic. XGBoost is the primary model because undercut success is an interaction problem: gap size, tyre offset, degradation, pit loss, and pace advantage combine in non-linear ways.

The key comparison is not just headline accuracy. The 2024 class balance makes it easy to score well by predicting fewer successes. Logistic Regression reaches higher accuracy and ROC-AUC, but XGBoost has the stronger success-class recall/F1 tradeoff, which is more useful when the goal is to catch real undercut opportunities rather than only maximize total correct labels.


In [9]:
# ─────────────────────────────────────────────
# Model 1: Logistic Regression (baseline)
# ─────────────────────────────────────────────

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE))
])
lr_pipe.fit(X_train, y_train)

lr_preds = lr_pipe.predict(X_test)
lr_probs = lr_pipe.predict_proba(X_test)[:, 1]

print('=== Logistic Regression (Baseline) ===')
print(f'Accuracy:  {accuracy_score(y_test, lr_preds):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, lr_probs):.3f}')
print()
print(classification_report(y_test, lr_preds, target_names=['No Gain', 'Position Gain']))

# Coefficients
coef_df = pd.DataFrame({
    'feature':     FEATURES,
    'coefficient': lr_pipe.named_steps['clf'].coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print('\nTop 10 logistic regression coefficients:')
print(coef_df.head(10).to_string(index=False))

=== Logistic Regression (Baseline) ===
Accuracy:  0.771
ROC-AUC:   0.825

               precision    recall  f1-score   support

      No Gain       0.82      0.86      0.84       236
Position Gain       0.63      0.56      0.59       100

     accuracy                           0.77       336
    macro avg       0.73      0.71      0.72       336
 weighted avg       0.76      0.77      0.77       336


Top 10 logistic regression coefficients:
              feature  coefficient
            gap_ahead    -2.326806
           pace_delta    -0.912971
         closing_rate     0.514976
compound_INTERMEDIATE     0.181194
         ca_deg_delta    -0.177860
      compound_MEDIUM    -0.164609
    pit_loss_fraction    -0.121317
             pit_loss    -0.116188
        compound_HARD     0.114124
        race_progress    -0.089391


In [10]:
# ─────────────────────────────────────────────
# Model 2: XGBoost (primary)
# ─────────────────────────────────────────────

# Balance classes
neg  = (y_train == 0).sum()
pos  = (y_train == 1).sum()
spw  = neg / pos if pos > 0 else 1.0

xgb_model = xgb.XGBClassifier(
    n_estimators      = 400,
    max_depth         = 4,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = spw,
    min_child_weight  = 5,
    random_state      = RANDOM_STATE,
    eval_metric       = 'logloss',
    verbosity         = 0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost (Primary) ===')
print(f'Accuracy:  {accuracy_score(y_test, xgb_preds):.3f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, xgb_probs):.3f}')
print()
print(classification_report(y_test, xgb_preds, target_names=['No Gain', 'Position Gain']))

# Save model
joblib.dump({'model': xgb_model, 'features': FEATURES}, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

=== XGBoost (Primary) ===
Accuracy:  0.738
ROC-AUC:   0.806

               precision    recall  f1-score   support

      No Gain       0.85      0.76      0.80       236
Position Gain       0.55      0.68      0.61       100

     accuracy                           0.74       336
    macro avg       0.70      0.72      0.71       336
 weighted avg       0.76      0.74      0.75       336

Model saved to /Users/cooperkerr/F1-Predictive-Analysis-Project/models/f1_undercut_model.pkl


In [11]:
# ─────────────────────────────────────────────
# Evaluation plots
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix — LR
ConfusionMatrixDisplay.from_predictions(
    y_test, lr_preds,
    display_labels=['No Gain', 'Position Gain'],
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title('Logistic Regression\nConfusion Matrix', fontsize=11)

# Confusion matrix — XGBoost
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_preds,
    display_labels=['No Gain', 'Position Gain'],
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('XGBoost\nConfusion Matrix', fontsize=11)

# ROC curves
RocCurveDisplay.from_predictions(y_test, lr_probs,  ax=axes[2], name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, xgb_probs, ax=axes[2], name='XGBoost')
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[2].set_title('ROC Curves — 2024 Holdout', fontsize=11)

plt.suptitle('Undercut Success Prediction — Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_model_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation saved to undercut_model_evaluation.png')

Evaluation saved to undercut_model_evaluation.png


## 7. SHAP Feature Attribution

Tree SHAP decomposes the XGBoost predictions into feature contributions. The bar chart ranks global importance by mean absolute SHAP value, while the beeswarm shows direction: high feature values in red, low values in blue. This is where I check whether the model's strongest signals line up with racing logic rather than just reporting an accuracy number.


In [12]:
# ─────────────────────────────────────────────
# SHAP feature importance
# ─────────────────────────────────────────────

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

plt.sca(axes[0])
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURES,
    show=False, max_display=12, plot_type='bar'
)
axes[0].set_title('Mean |SHAP| — Feature Importance', fontsize=11)

plt.sca(axes[1])
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURES,
    show=False, max_display=12
)
axes[1].set_title('SHAP Beeswarm — Direction & Magnitude', fontsize=11)

plt.suptitle('XGBoost SHAP Analysis — Drivers of Undercut Success', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'undercut_shap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('SHAP plot saved to undercut_shap.png')

SHAP plot saved to undercut_shap.png


## 8. Probability Calibration

For a strategy model, a predicted probability is only useful if it roughly matches observed frequency. This calibration view bins XGBoost probabilities and compares each bin's average prediction to its actual 2024 success rate. Points near the diagonal mean the model's probability scale is meaningful; points above or below it show over- or under-confidence.


In [13]:
# ─────────────────────────────────────────────
# Probability calibration: P(success) vs actual rate
# ─────────────────────────────────────────────

prob_bins = np.linspace(0, 1, 11)
bin_indices = np.digitize(xgb_probs, prob_bins) - 1
bin_indices = np.clip(bin_indices, 0, len(prob_bins) - 2)

cal_df = pd.DataFrame({'prob': xgb_probs, 'actual': y_test, 'bin': bin_indices})
cal_stats = cal_df.groupby('bin').agg(
    mean_prob=('prob',   'mean'),
    actual_rate=('actual', 'mean'),
    n=('actual', 'count')
).reset_index()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(cal_stats['mean_prob'], cal_stats['actual_rate'],
           s=cal_stats['n'] * 3, alpha=0.8, color='steelblue', label='Model bins')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Actual Success Rate')
ax.set_title('XGBoost Probability Calibration\n(bubble size ∝ sample count)', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('undercut_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# ─────────────────────────────────────────────
# Model comparison summary
# ─────────────────────────────────────────────

from sklearn.metrics import precision_score, recall_score, f1_score

summary = pd.DataFrame([
    {
        'Model':     'Logistic Regression (baseline)',
        'Accuracy':  round(accuracy_score(y_test, lr_preds),  3),
        'ROC-AUC':   round(roc_auc_score(y_test, lr_probs),   3),
        'Precision': round(precision_score(y_test, lr_preds), 3),
        'Recall':    round(recall_score(y_test, lr_preds),    3),
        'F1':        round(f1_score(y_test, lr_preds),        3),
    },
    {
        'Model':     'XGBoost (primary)',
        'Accuracy':  round(accuracy_score(y_test, xgb_preds),  3),
        'ROC-AUC':   round(roc_auc_score(y_test, xgb_probs),   3),
        'Precision': round(precision_score(y_test, xgb_preds), 3),
        'Recall':    round(recall_score(y_test, xgb_preds),    3),
        'F1':        round(f1_score(y_test, xgb_preds),        3),
    },
])

print('=== Final Model Comparison (2024 Holdout) ===')
print(summary.to_string(index=False))

=== Final Model Comparison (2024 Holdout) ===
                         Model  Accuracy  ROC-AUC  Precision  Recall    F1
Logistic Regression (baseline)     0.771    0.825      0.629    0.56 0.593
             XGBoost (primary)     0.738    0.806      0.548    0.68 0.607


---
## Key Findings

**Base rate.** Undercuts are more viable than overcuts in this sample, but they are still not automatic. Across 2022-2024 the base-rate success was **28.8%**. In the 2024 holdout specifically it was **29.8%**. So an "always no gain" baseline would already score about **70.2%** accuracy on the 2024 holdout. That is why I treat accuracy as only one part of the evaluation and look closely at recall and F1 for actual position-gain cases.

**Predictive accuracy, 2024 holdout.**
| Model | Accuracy | ROC-AUC | Interpretation |
|---|---:|---:|---|
| Logistic Regression | 0.771 | 0.825 | Higher headline accuracy and ROC-AUC, but more conservative about predicting successful undercuts. |
| XGBoost (primary) | 0.738 | 0.806 | Lower accuracy and ROC-AUC, but stronger recall/F1 on true successes. |

The confusion matrices make the tradeoff clear. Logistic Regression gets more total labels right because it is more cautious, especially on the majority/no-gain side. XGBoost catches more actual position-gain cases, which lowers overall accuracy but improves the model's usefulness as a strategy screen. In this setting, missing a viable undercut is a different error than flagging one too aggressively, so recall and F1 on the success class are important alongside accuracy and ROC-AUC.

**Dominant features (SHAP mean |φ|).**
1. **`gap_ahead`** (1.34), the time gap the undercutting driver must recover. This is the strongest signal: even with fresh tyres, the strategy has to fit inside the available pit-cycle margin.
2. **`pace_delta`** (0.75), the undercutting driver's recent pace relative to the car ahead. If the pitting driver already has stronger pace, the fresh-tyre phase is much more likely to convert into track position.
3. **`ca_deg_delta`** (0.33), the degradation slope for the car ahead. A vulnerable target car whose tyres are dropping away makes the undercut more attractive.
4. **`closing_rate`** (0.30), whether the gap was already shrinking before the stop. A driver who is actively closing is better positioned to turn the pit stop into an overtake.
5. **`deg_delta`** (0.28), the undercutting driver's own degradation slope. This helps distinguish a driver who needs fresh tyres because their current stint is fading from one who may not gain enough by stopping early.

**Interpretation of the feature pattern.** The model is not simply learning "newer tyres good." The strongest SHAP signal is still the battle geometry: how large the gap is and whether the pitting driver has enough live pace to erase it. Tyre-age and degradation variables matter because they explain why that pace gap may change after the stop, but the undercut only works when the timing window is realistic.

**Calibration.** The calibration plot is a probability sanity check rather than a second accuracy metric. If the points track the diagonal, the XGBoost probabilities can be read as approximate undercut-success chances. Where bins drift away from the diagonal, the model may still rank attempts correctly but should be treated carefully as an exact probability estimator.

**Interpretation.** There is still a real error floor here, and that is expected. Mechanical issues, sudden rain, radio communication, traffic, driver execution, and team strategy calls all add noise that a lap-timing model cannot fully observe. A useful model should estimate undercut probability under observable race-state conditions, not pretend the outcome is deterministic.
